# pyFM - Basic Functions

This notebook walks through the core functionality of pyFM: loading and
inspecting triangle meshes, computing their spectral decomposition, optimizing
a functional map between two shapes, refining it, and evaluating accuracy
against a ground-truth correspondence.

In [ ]:
import sys

sys.path.append("../../")

In [ ]:
import numpy as np

from pyFM.mesh import TriMesh
from pyFM.functional import FunctionalMapping

import meshplot as mp


def plot_mesh(myMesh, cmap=None):
    mp.plot(myMesh.vertlist, myMesh.facelist, c=cmap)


def double_plot(myMesh1, myMesh2, cmap1=None, cmap2=None):
    d = mp.subplot(myMesh1.vertlist, myMesh1.facelist, c=cmap1, s=[2, 2, 0])
    mp.subplot(myMesh2.vertlist, myMesh2.facelist, c=cmap2, s=[2, 2, 1], data=d)


def visu(vertices):
    min_coord, max_coord = np.min(vertices, axis=0, keepdims=True), np.max(
        vertices, axis=0, keepdims=True
    )
    cmap = (vertices - min_coord) / (max_coord - min_coord)
    return cmap

# 2 - Loading and processing a mesh

### Basic mesh methods

A `TriMesh` can be created from a file path (`.off` or `.obj`) or directly from vertex and face arrays. When loading, the mesh can be centered,
area-normalized, rotated, or translated.

Vertices and faces live in `mesh.vertlist` and `mesh.facelist`; `mesh.vertices`
and `mesh.faces` are equivalent aliases. The longer names are intentional —
they prevent naming collisions in functions where a local variable called
`vertices` would shadow the attribute.

Derived quantities (edges, per-face and per-vertex areas, normals, …) are
computed lazily and cached on first access.

In [ ]:
mesh1 = TriMesh("../data/cat-00.off", area_normalize=True, center=False)
mesh2 = TriMesh(mesh1.vertlist, mesh1.facelist)

In [ ]:
# Attributes are computed on the fly and cached
edges = mesh1.edges

area = mesh1.area

face_areas = mesh1.face_areas
vertex_areas = mesh1.vertex_areas
face_normals = mesh1.normals

# Area-weighted vertex normals (default)
vertex_normals_a = mesh1.vertex_normals

# Switch to uniform weighting
mesh1.set_vertex_normal_weighting("uniform")
vertex_normals_u = mesh1.vertex_normals

### Geodesics

Four options are available for computing geodesic distances:

- **potpourri3d heat method** — backed by a robust Laplacian, recommended for most use cases
- **pyFM heat method** — pure-Python implementation, slower but fully transparent
- **Dijkstra** — graph-based approximation, fast but less accurate on coarse meshes
- **potpourri3d fast marching** - standard algorithm for geodesic distances

In [ ]:
# Geodesic distance from a single vertex
# Set robust=False to use the pure-Python heat method instead of potpourri3d
dists = mesh1.geod_from(1000, robust=True)

In [ ]:
# I recommend sticking to the default values here.
S1_geod = mesh1.get_geodesic(verbose=True)

### Laplacian and spectral decomposition

The LBO spectrum is computed with `mesh.process(k=...)`, which stores results
in `mesh.eigenvalues` and `mesh.eigenvectors`.

`mesh.project` and `mesh.unproject` convert a function between the full vertex
domain and the truncated spectral basis.

Gradient and divergence operators are also available.

Norms can be computed via `mesh.l2_sqnorm` (squared $L^2$) and
`mesh.h1_sqnorm` (squared $H^1_0$).

In [ ]:
# By default does not use the intrinsic Delaunay Laplacian.
# For functional map methods, I'd recommend setting intrinsic=True.
mesh1.process(k=100, intrinsic=False, verbose=True)

In [ ]:
# plot the third eigenfunction
plot_mesh(mesh1, mesh1.eigenvectors[:, 2])

# 3 - Computing the functional map

### Loading meshes

In [ ]:
mesh1 = TriMesh("../data/cat-00.off")
mesh2 = TriMesh("../data/lion-00.off")
print(
    f"Mesh 1 : {mesh1.n_vertices:4d} vertices, {mesh1.n_faces:5d} faces\n"
    f"Mesh 2 : {mesh2.n_vertices:4d} vertices, {mesh2.n_faces:5d} faces"
)

double_plot(mesh1, mesh2)

### Descriptors and preprocessing

In [ ]:
process_params = {
    "K": (35, 35),  # number of eigenvalues on source and target
    "landmarks": np.loadtxt("../data/landmarks.txt", dtype=int)[:5],  # 5 landmarks
    "subsample_step": 5,  # use every 5th descriptor to keep things light
    "descr_type": "WKS",  # WKS or HKS
}

model = FunctionalMapping(mesh1, mesh2)
model.preprocess(**process_params, verbose=True)

### Fitting the map

$\renewcommand{\RR}{\mathbb{R}}$
$\renewcommand{\Ss}{\mathcal{S}}$
$\renewcommand{\uargmin}[1]{\underset{#1}{\text{argmin}}\;}$
$\renewcommand{\uargmax}[1]{\underset{#1}{\text{argmax}}\;}$
$\def\*#1{\mathbf{#1}}$

In pyFM, functional maps $\*C:\Ss_1\to\Ss_2$ and pointwise maps $T:\Ss_2\to\Ss_1$ always go in opposite directions, with $\*C$ going from shape 1 to shape 2.

The optimization problem is
$$
\uargmin{\*C\in\RR^{k_2\times k_1}}
  w_{\text{descr}}\|\*C\*A - \*B\|^2
+ w_{\text{lap}}\|\*C\Delta_1 - \Delta_2\*C\|^2
+ w_{\text{d-comm}}\sum_i \|\*C\Gamma_1^i - \Gamma_2^i\*C\|^2
+ w_{\text{orient}}\sum_i \|\*C\Lambda_1^i - \Lambda_2^i\*C\|^2
$$
where $\Gamma^i$ are [multiplicative operators](http://www.lix.polytechnique.fr/~maks/papers/fundescEG17.pdf) and $\Lambda^i$ are [orientation-preserving operators](https://arxiv.org/abs/1806.04455) associated to the $i$-th descriptor.


In [ ]:
fit_params = {"w_descr": 1e0, "w_lap": 1e-2, "w_dcomm": 1e-1, "w_orient": 0}


model.fit(**fit_params, verbose=True)

### Pointwise map and visualization


In [ ]:
p2p_21 = model.get_p2p(n_jobs=1)
cmap1 = visu(mesh1.vertlist)
cmap2 = cmap1[p2p_21]
double_plot(mesh1, mesh2, cmap1, cmap2)

# 4 - Refining the Functional Map

`model.FM` returns the current functional map. Use the methods below to refine it.

### ICP

In [ ]:
FM_12_icp = model.icp_refine(verbose=True)
p2p_21_icp = model.get_p2p(FM_12_icp)
cmap1 = visu(mesh1.vertlist)
cmap2 = cmap1[p2p_21_icp]
double_plot(mesh1, mesh2, cmap1, cmap2)

### ZoomOut

In [ ]:
FM_12_zo = model.zoomout_refine(nit=10, step=5, verbose=True)
print(FM_12_zo.shape)
p2p_21_zo = model.get_p2p(FM_12_zo)
cmap1 = visu(mesh1.vertlist)
cmap2 = cmap1[p2p_21_zo]
double_plot(mesh1, mesh2, cmap1, cmap2)

# 5 - Evaluating Results

In [ ]:
import pyFM.eval

In [ ]:
# Compute geodesic distance matrix on the cat mesh
A_geod = mesh1.get_geodesic(verbose=True)

In [ ]:
# Load an approximate ground truth map
gt_p2p = np.loadtxt("../data/lion2cat", dtype=int)

acc_base = pyFM.eval.accuracy(p2p_21, gt_p2p, A_geod, sqrt_area=mesh1.sqrtarea)

acc_icp = pyFM.eval.accuracy(p2p_21_icp, gt_p2p, A_geod, sqrt_area=np.sqrt(mesh1.area))

acc_zo = pyFM.eval.accuracy(p2p_21_zo, gt_p2p, A_geod, sqrt_area=np.sqrt(mesh1.area))

print(
    f"Accuracy results\n"
    f"\tBasic FM : {1e2*acc_base:.2f}\n"
    f"\tICP refined : {1e2*acc_icp:.2f}\n"
    f"\tZoomOut refined : {1e2*acc_zo:.2f}\n"
)